# CMSC · AI ACOUSTIC V1 — Notebook Maestro (DISEÑO)

**Misión:** CMSC_ACOUSTIC_DATASET_AND_MODEL_V1 · **FASE 6**
**Fecha:** 2026-10-02 · **HEAD:** `83b6089` · **Estado:** SOLO DISEÑO

> ## ⚠️ NO EJECUTAR
> Este notebook **no se ha ejecutado** y **no debe ejecutarse** sin una misión explícita de implementación
> autorizada por Bernardo. No descarga datos, no entrena y no produce artefactos por sí mismo.
> Su función es **congelar el pipeline reproducible** que materializará:

1. Carga del dataset `SIGCT_RURAL_ACOUSTIC_DATASET_V1` (`acoustic_labels_v1.csv` + WAV).
2. Extracción/agregación de features `acoustic.feat.v2` → `acoustic.feat.window.v2` (44 dims).
3. Benchmark de modelos (MA-0..MA-7) con métricas comparables a M1/M2.
4. Calibración de confianza (ECE).
5. Selección de **Model V1** (`ml-v1-tabular`) y export a ONNX.
6. Artefactos de gobernanza: `validation_metrics.json` + `model_card.md`.

**Documentos rectores:**
- `docs/AI_ACOUSTIC_FEATURE_MAP.md`
- `docs/AI_ACOUSTIC_DATASET_V1_DESIGN.md`
- `docs/AI_ACOUSTIC_MODEL_BENCHMARK.md`
- `docs/AI_ACOUSTIC_S81_INTEGRATION.md`

**Invariantes:** no inventar features fuera de las 22 verificadas; no declarar métricas sin artefacto;
`seed=42`; group split por `(session_id, place_id)`.
> **Nota de esquema (2026-10-05).** Este notebook usa `acoustic.feat.v2` / `acoustic.feat.window.v2`,
> que son los esquemas vigentes. `acoustic.feat.v1` / `window.v1` quedan **CONGELADOS**: su feature 21
> (`profileEntropy`) usaba `db[i]²` en lugar de potencia lineal y truncaba 32 de los 2048 bins del
> espectro, de modo que daba entropía ≈1 tanto al silencio como al ruido blanco. No usarlo para entrenar.
> Detalle y cifras: `docs/AI_ACOUSTIC_FEATURE_MAP.md` §5. El notebook sigue **sin ejecutar**: no hay
> dataset (`metadata.csv` tiene 0 filas) ni modelo. S81 permanece en `DISENO`.


## 0. Pipeline (vista general)

```
WAV (3 s, mono, 44.1k)  ->  features por frame (acoustic.feat.v2, 22)  ->  agregación (window.v2, 44)
        ->  clasificador (tabular / CNN / transfer)  ->  calibración  ->  S81 (label, confidence, method)
        ->  validation_metrics.json + model_card.md  ->  gate G-M-*  ->  estado REAL-LOCAL
```

Este notebook cubre el tramo **offline de investigación**. La inferencia en navegador de S81 se diseña en
`docs/AI_ACOUSTIC_S81_INTEGRATION.md`.

## 1. Configuración (rutas, seed, esquema, clases)

In [ ]:
# DISEÑO — no ejecutar sin autorización.
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 42
DATASET_ROOT = Path("data/datasets/sigct_rural_acoustic/v1")
LABELS_CSV = DATASET_ROOT / "labels/acoustic_labels_v1.csv"
FEATURES_CSV = DATASET_ROOT / "features/acoustic_features_v1.csv"  # cache opcional
RUNS_DIR = Path("benchmark/runs/MA_acoustic_v1")
ARTIFACTS_DIR = Path("docs/ai/model_cards")

CLASSES = [
    "silencio", "viento", "lluvia", "agua", "insectos",
    "ave", "voz", "motor", "ruido_desconocido",
]

FEATURE_SCHEMA_V2 = [
    "dominantHzNorm", "peakDbNorm", "centroidNorm", "flatness", "rms", "logEnergy", "levelDb",
    "bandPctLow", "bandPctMid", "bandPctHigh",
    "bandPeakLowNorm", "bandPeakMidNorm", "bandPeakHighNorm",
    "bandPeakDbLow", "bandPeakDbMid", "bandPeakDbHigh",
    "spectralBandwidthNorm", "spectralRolloffNorm", "zeroCrossingRate",
    "crestFactor", "spectralFlux", "profileEntropy",
]
assert len(FEATURE_SCHEMA_V2) == 22, "acoustic.feat.v2 debe tener 22 features"

# Agregación temporal (window.v2): mean + std por feature -> 44 dims.
AGG_STATS = ["mean", "std"]
WINDOW_SCHEMA_V2 = [f"{f}__{s}" for f in FEATURE_SCHEMA_V2 for s in AGG_STATS]
assert len(WINDOW_SCHEMA_V2) == 44
np.random.seed(SEED)
print("Esquema window.v2:", len(WINDOW_SCHEMA_V2), "dims")

## 2. Carga y auditoría del dataset

Se espera `acoustic_labels_v1.csv` con las columnas de `AI_ACOUSTIC_DATASET_V1_DESIGN.md` §A.6.
Si el archivo no existe, el notebook **se detiene** (no fabrica datos: regla de honestidad).

In [ ]:
# DISEÑO — no ejecutar sin autorización.
if not LABELS_CSV.exists():
    raise SystemExit(
        "Dataset no materializado: falta acoustic_labels_v1.csv. "
        "Ver AI_ACOUSTIC_DATASET_V1_DESIGN.md (FASE 2/3). No se fabrican datos."
    )

labels = pd.read_csv(LABELS_CSV)
required_cols = {
    "sample_id", "file", "sha256", "duration_s", "sample_rate", "channels",
    "class_id", "class_name", "contaminated", "place_id", "environment",
    "session_id", "device_id", "timestamp", "license", "source", "annotator_id",
}
missing = required_cols - set(labels.columns)
assert not missing, f"Columnas faltantes: {missing}"

print("Total clips:", len(labels))
print("\nConteo por clase:")
print(labels["class_id"].value_counts().reindex(CLASSES))
print("\nPor entorno:")
print(labels["environment"].value_counts())
print("\nPor fuente:")
print(labels["source"].value_counts())

# Gate G-DS-2: >=150 clips/clase propios
counts = labels["class_id"].value_counts().reindex(CLASSES).fillna(0).astype(int)
min_ok = counts.min() >= 150
print("\nG-DS-2 (>=150/clase):", "PASS" if min_ok else "FAIL -> no entrenar todavia")

## 3. Features `acoustic.feat.v2` → `acoustic.feat.window.v2`

El extractor canónico vive en JS (`pages/cmsc/acoustic/featureExtractor.js`, a crear).
Aquí se **consume** la caché `acoustic_features_v1.csv` (una fila por frame) para garantizar que el
esquema es el mismo. Alternativa: recomputar desde WAV con una implementación espejo validada por un
**test golden** contra el vector JS.

In [ ]:
# DISEÑO — no ejecutar sin autorización.
def aggregate_window(frame_features: pd.DataFrame) -> pd.Series:
    """mean+std de las 22 features de un clip -> 44 dims (acoustic.feat.window.v2)."""
    mat = frame_features[FEATURE_SCHEMA_V2].to_numpy(dtype=np.float64)
    stats = np.vstack([mat.mean(axis=0), mat.std(axis=0)]).T.reshape(-1)  # (22,2)
    return pd.Series(stats, index=WINDOW_SCHEMA_V2)


def build_window_table(labels_df: pd.DataFrame, features_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sample_id, grp in features_df.groupby("sample_id"):
        vec = aggregate_window(grp).to_dict()
        vec["sample_id"] = sample_id
        rows.append(vec)
    X = pd.DataFrame(rows).set_index("sample_id")
    y = labels_df.set_index("sample_id")["class_id"].reindex(X.index)
    return X, y


# frame-features esperado: columnas [sample_id, frame_index] + FEATURE_SCHEMA_V2
# X, y = build_window_table(labels, pd.read_csv(FEATURES_CSV))
print("Pipeline window.v2 definido (no ejecutado).")

## 4. Split anti-fuga (group split)

El split **no** es aleatorio sobre clips: agrupa por `(session_id, place_id)` (regla FASE 2 §A.8).
Los manifiestos `split_lists/*.csv` son la fuente de verdad; este notebook solo los consume.

In [ ]:
# DISEÑO — no ejecutar sin autorización.
def load_split(name: str) -> pd.DataFrame:
    return pd.read_csv(DATASET_ROOT / f"split_lists/{name}.csv")


# train_ids = load_split("train")["sample_id"]
# validation_ids = load_split("validation")["sample_id"]
# test_ids = load_split("test")["sample_id"]
# assert not (set(train_ids) & set(test_ids)), "Fuga detectada: train/intersect test"
print("Split por manifiestos (group split). No se parte aleatoriamente.")

## 5. Baselines tabulares (MA-0 .. MA-4)

In [ ]:
# DISEÑO — no ejecutar sin autorización.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CANDIDATES = {
    "MA-1_logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "MA-2_randomforest": RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1),
    "MA-3_histgb": HistGradientBoostingClassifier(random_state=SEED),
    "MA-4_mlp": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), random_state=SEED, max_iter=1000)),
}
# MA-0 rules-v0 se implementa aparte (umbrales deterministas, ver ARCHITECTURE_V1).
print("Candidatos tabulares:", list(CANDIDATES))

## 6. Modelos profundos (MA-5 CNN, MA-6 YAMNet, MA-7 AST/PANNs) — opcional

Solo si el volumen de datos lo justifica (>=300/clase) y el gate MA-3 está cerrado.
Se documentan aquí para trazabilidad; **no** forman parte de V1.

In [ ]:
# DISEÑO — no ejecutar sin autorización.
# MA-5: CNN sobre log-mel (64 mel x T frames) — PyTorch.
# MA-6: YAMNet (TF Hub, 3.7M params) -> embeddings 1024 -> cabeza lineal. Export TFLite.
# MA-7: AST / PANNs CNN14 / ConvNeXt — alto costo, solo servidor (mAP AudioSet ~0.43-0.47).
DEEP_NOTE = "VER AI_ACOUSTIC_MODEL_BENCHMARK.md secciones 3 y 6.2 (Nivel B)."
print(DEEP_NOTE)

## 7. Evaluación (métricas comparables a M1/M2)

In [ ]:
# DISEÑO — no ejecutar sin autorización.
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix


def expected_calibration_error(y_true_idx, probs, n_bins=15):
    """ECE multiclase (reutiliza la lógica de benchmark/src/metrics.py)."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true_idx).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def evaluate(model, X_test, y_test_idx):
    probs = model.predict_proba(X_test)
    pred = probs.argmax(axis=1)
    return {
        "macro_f1": float(f1_score(y_test_idx, pred, average="macro")),
        "balanced_accuracy": float(balanced_accuracy_score(y_test_idx, pred)),
        "ece": expected_calibration_error(y_test_idx, probs),
        "confusion_matrix": confusion_matrix(y_test_idx, pred).tolist(),
    }
print("Evaluación definida: macro_f1, balanced_accuracy, ECE, confusion_matrix.")

## 8. Calibración de confianza

`confidence` de S81 debe ser una probabilidad **calibrada**. Gate: ECE <= 0.10.

In [ ]:
# DISEÑO — no ejecutar sin autorización.
from sklearn.calibration import CalibratedClassifierCV

def calibrate(base_model, method="isotonic"):
    # method: 'sigmoid' (Platt) o 'isotonic'
    return CalibratedClassifierCV(base_model, method=method, cv=5)
print("Calibración: CalibratedClassifierCV (sigmoid | isotonic). Bloqueo si ECE>0.15.")

## 9. Selección de Model V1 y export

**V1 = `ml-v1-tabular`** (MA-3 HistGB o MA-1 LogReg) según `AI_ACOUSTIC_MODEL_BENCHMARK.md` §6.
Export a ONNX (carga dinámica en cliente; limite 15 MB).

In [ ]:
# DISEÑO — no ejecutar sin autorización.
# from skl2onnx import to_onnx
# onx = to_onnx(best_model, X_train.iloc[:1].astype(np.float32).to_numpy())
# (RUNS_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
# (RUNS_DIR / "checkpoints/best.onnx").write_bytes(onx.SerializeToString())
print("Export destino:", RUNS_DIR / "checkpoints/best.onnx", "(no ejecutado).")

## 10. Artefactos de gobernanza

Ningún modelo "existe" sin `validation_metrics.json` y `model_card.md`.

In [ ]:
# DISEÑO — no ejecutar sin autorización.
def write_validation_metrics(metrics: dict, out: Path):
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")


# write_validation_metrics(metrics, RUNS_DIR / "validation_metrics.json")
# model_card = f"""# Acoustic Classifier V1\n\n- schema: acoustic.feat.window.v2 (44)\n- dataset: sigct_rural_acoustic/v1\n- method: ml-v1\n- metrics: {metrics}\n- state: REAL-LOCAL (gate G-M-2)\n"""
# (ARTIFACTS_DIR / "acoustic_clf_v1.md").write_text(model_card, encoding="utf-8")
print("Artefactos definidos:", RUNS_DIR / "validation_metrics.json", "|", ARTIFACTS_DIR / "acoustic_clf_v1.md")

## 11. Gates de honestidad y cierre

| Gate | Criterio | Efecto |
|---|---|---|
| G-DS-2 | >=150 clips/clase propios | habilita entrenamiento |
| G-M-2 | `ml-v1-tabular` supera `rules-v0` (macro-F1 y ECE) | S81 = `REAL-LOCAL` |
| G-M-3 | ECE <= 0.10 | confianza creíble |
| G-M-4 | latency <= 250 ms y size <= 15 MB | viable offline |
| G-M-5 | leave-one-place-out >= umbral | generaliza de sitio |

> **Recordatorio final:** mientras no existan dataset propio (**G-DS-2**) ni artefactos de modelo
> (`validation_metrics.json` + `model_card.md`), S81 permanece en **`DISENO`**. Este notebook es el plano,
> no la prueba.